# 第41章 小提琴图（violinplot）

用小提琴图展示分类组的平滑密度形状和中心信息。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

样本量足够，希望比较多峰、偏态或尾部形状。

## 数据结构

每组包含较多连续数值观察。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 inner="quart" 改为 inner="box"，对比四分位线与箱线摘要的显示效果
2. 调整 bw_adjust 参数（如 0.5 或 2.0），说明带宽对密度曲线平滑度的影响
3. 修改 cut 参数从 0 为 2，观察密度曲线在数据范围外的延伸变化


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


sns.set_theme(style="whitegrid", context="notebook")
diamonds = pd.read_csv(f"{base_url}/datasets/diamonds.csv")
orders_full = diamonds.assign(
    category = diamonds["cut"], channel=diamonds["color"], region=diamonds["clarity"],
    order_value = diamonds["price"], items=diamonds["carat"],
    satisfied=np.where(diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"),
)
orders = orders_full.sample(2_000, random_state=36).copy()
taxis = pd.read_csv(f"{base_url}/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
marketing_full = taxis.assign(
    channel = taxis["payment"].fillna("unknown"), visits=taxis["distance"],
    ad_spend = taxis["tip"], sales=taxis["total"],
    conversion = (taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(min(2_000, len(marketing_full)), random_state=36).copy()
flights = pd.read_csv(f"{base_url}/datasets/flights.csv")
daily = flights.assign(
    date = pd.to_datetime(flights["year"].astype("string") + "-" + flights["month"] + "-01"),
    region = "AirPassengers", sales=flights["passengers"],
)
print(f"Diamonds：{len(diamonds):,} 行；NYC Taxis：{len(taxis):,} 行；Flights：{len(flights):,} 行")
print("图表兼容列均由公开数据原始字段直接映射；高成本图使用固定 2,000 行样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
sns.violinplot(data=orders, x="category", y="order_value", hue="category", palette="Set2", legend=False, inner="quart", cut=0, ax=ax)
ax.set(title="品类客单价密度", xlabel="品类", ylabel="客单价（元）")
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
two_channels = orders[orders["channel"].isin(["自然流量", "广告"])]
fig, ax = plt.subplots(figsize=(9, 4.8))
sns.violinplot(data=two_channels, x="category", y="order_value", hue="channel", split=True, inner="quart", cut=0, palette=["#1a73e8", "#f9ab00"], ax=ax)
ax.set(title="自然流量与广告客单价密度", xlabel="品类", ylabel="客单价（元）")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()


## 3. 参数说明

- inner：内部摘要
- cut：密度延伸
- bw_adjust：带宽
- split：二分类对称拆分


## 4. 结果解读

宽处表示估计密度较高；形状受带宽影响，不等于实际频数。


## 常见误区

- 小样本产生误导性平滑形状
- 不说明每组样本量
- 不同组独立归一化却比较绝对宽度


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.violinplot(data=orders, x="region", y="items", hue="region", legend=False, inner="box", cut=0, palette="pastel", ax=ax)
ax.set(title="区域购买件数分布", xlabel="区域", ylabel="件数")
fig.tight_layout()
plt.show()


## 本章小结

用小提琴图展示分类组的平滑密度形状和中心信息。


### 你已经掌握

- 判断小提琴图（violinplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 样本量足够，希望比较多峰、偏态或尾部形状。 |
| 数据结构 | 每组包含较多连续数值观察。 |
| 结果解读 | 宽处表示估计密度较高；形状受带宽影响，不等于实际频数。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `inner` | 内部摘要 |
| `cut` | 密度延伸 |
| `bw_adjust` | 带宽 |
| `split` | 二分类对称拆分 |


### 需要注意

- 小样本产生误导性平滑形状
- 不说明每组样本量
- 不同组独立归一化却比较绝对宽度


### 完成检查

- [ ] 能判断什么问题适合使用小提琴图（violinplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
